# Train Baseline codeBERT:

In [1]:
import os
import gc
import sys
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

In [2]:
SEED = 42
BATCH_SIZE = 16
NUM_EPOCHS = 3
LR = 2e-5
MODEL_NAME = "microsoft/codebert-base"

TARGET_DIR = "/kaggle/input/datasets/ppark8798/smartbug/"
SAVE_DIR = "/kaggle/working/"
os.makedirs(SAVE_DIR, exist_ok=True)

FREEZE_LAYERS = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [4]:
script_path = "/kaggle/input/models/ppark8798/baseline-model/pytorch/default/7"
if script_path not in sys.path:
    sys.path.append(script_path)

import baseline
import smart_datasets

In [5]:
train_raw = torch.load(os.path.join(TARGET_DIR, "train_chunks_subset.pt"))
train_dataset = smart_datasets.SmartContractDataset(train_raw)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
del train_raw
gc.collect()

8

In [6]:
val_raw = torch.load(os.path.join(TARGET_DIR, "val_chunks.pt"))
val_dataset = smart_datasets.SmartContractDataset(val_raw)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
del val_raw
gc.collect()

8

# Define Model

In [7]:
model = baseline.BaselineModel(model_name=MODEL_NAME).to(device)

# freezing
if FREEZE_LAYERS > 0:
    # freeze embeddings
    for param in model.encoder.embeddings.parameters():
        param.requires_grad = False

    # freeze first N layers
    for layer in model.encoder.encoder.layer[:FREEZE_LAYERS]:
        for param in layer.parameters():
            param.requires_grad = False
optimizer = optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR
)
criterion = nn.BCEWithLogitsLoss(reduction="none")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

# Train Function

In [8]:
def train_epoch():
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc="Training"):
        # anchor
        anchor_ids = batch["anchor_input_ids"].to(device)
        anchor_mask = batch["anchor_attention_mask"].to(device)

        # positive
        pos_ids = batch["pos_input_ids"].to(device)
        pos_mask = batch["pos_attention_mask"].to(device)

        # labels + weights
        labels = batch["label"].to(device).float()
        weights = batch["weight"].to(device).float()

        optimizer.zero_grad(set_to_none=True)

        # forward
        anchor_logits = model(anchor_ids, anchor_mask).view(-1)
        pos_logits = model(pos_ids, pos_mask).view(-1)
        assert anchor_logits.shape == labels.shape

        # loss
        loss_anchor = criterion(anchor_logits, labels)
        loss_pos = criterion(pos_logits, labels)
        loss = ((loss_anchor + loss_pos) / 2 * weights).mean()

        # backward
        loss.backward()
        params = [p for p in model.parameters() if p.requires_grad]
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [9]:
def evaluate():
    model.eval()

    contract_scores = defaultdict(list)
    contract_labels = {}

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch["anchor_input_ids"].to(device)
            mask = batch["anchor_attention_mask"].to(device)
            labels = batch["label"]
            contract_ids = batch["contract_id"]

            logits = model(input_ids, mask)
            probs = torch.sigmoid(logits).cpu()

            for i in range(len(probs)):
                cid = int(contract_ids[i])
                contract_scores[cid].append(float(probs[i]))
                contract_labels[cid] = int(labels[i])

    # max pooling per contract
    contract_probs = []
    contract_true = []

    for cid in sorted(contract_scores.keys()):
        contract_probs.append(max(contract_scores[cid]))
        contract_true.append(contract_labels[cid])

    contract_probs = np.array(contract_probs)
    contract_true = np.array(contract_true)
    preds = (contract_probs > 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(contract_true, preds),
        "precision": precision_score(contract_true, preds, zero_division=0),
        "recall": recall_score(contract_true, preds, zero_division=0),
        "f1": f1_score(contract_true, preds, zero_division=0),
    }

    try:
        metrics["pr_auc"] = average_precision_score(contract_true, contract_probs)
    except:
        metrics["pr_auc"] = 0.0

    try:
        metrics["roc_auc"] = roc_auc_score(contract_true, contract_probs)
    except:
        metrics["roc_auc"] = 0.0

    print("\nValidation Metrics")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    return metrics

In [10]:
best_pr_auc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====")

    loss = train_epoch()
    print(f"Train Loss: {loss:.4f}")

    # save checkpoint
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
    }, os.path.join(SAVE_DIR, f"checkpoint_{epoch+1}.pt"))

    metrics = evaluate()

    if metrics["pr_auc"] > best_pr_auc:
        best_pr_auc = metrics["pr_auc"]

        torch.save({
            "model_state_dict": model.state_dict(),
            "metrics": metrics
        }, os.path.join(SAVE_DIR, "best_model.pt"))

        print(f"New best PR-AUC: {best_pr_auc:.4f}")

    torch.cuda.empty_cache()

print("\nTraining complete.")
print(f"Best PR-AUC: {best_pr_auc:.4f}")


===== Epoch 1/3 =====


Training:   0%|          | 0/3227 [00:00<?, ?it/s]

Train Loss: 0.5435


Evaluating:   0%|          | 0/2076 [00:00<?, ?it/s]


Validation Metrics
accuracy: 0.6538
precision: 0.6216
recall: 0.9626
f1: 0.7554
pr_auc: 0.7014
roc_auc: 0.7379
New best PR-AUC: 0.7014

===== Epoch 2/3 =====


...


Validation Metrics
accuracy: 0.6319
precision: 0.6035
recall: 0.9826
f1: 0.7478
pr_auc: 0.7316
roc_auc: 0.7305
New best PR-AUC: 0.7316

===== Epoch 3/3 =====


Training:   0%|          | 0/3227 [00:00<?, ?it/s]

Train Loss: 0.3616


Evaluating:   0%|          | 0/2076 [00:00<?, ?it/s]


Validation Metrics
accuracy: 0.6494
precision: 0.6167
recall: 0.9740
f1: 0.7552
pr_auc: 0.7099
roc_auc: 0.7274

Training complete.
Best PR-AUC: 0.7316
